In [11]:
import json
import asyncio
from typing import List, Dict, Union
from httpx import AsyncClient, Response
from parsel import Selector
from loguru import logger as log
import pandas as pd

In [16]:
# initialize an async httpx client
client = AsyncClient(
    # enable http2
    http2=True,
    # add basic browser like headers to prevent getting blocked
    headers={
        "Accept-Language": "en-US,en;q=0.9",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Cookie": "intl_splash=false"
    },
    follow_redirects=True
)

In [19]:
def parse_subreddit(response: Response) -> List[Dict]:
    """Parse article data from HTML"""
    selector = Selector(response.text)
    url = str(response.url)
    info = {}
    info["id"] = url.split("/r")[-1].replace("/", "")
    info["description"] = selector.xpath("//shreddit-subreddit-header/@description").get()
    members = selector.xpath("//shreddit-subreddit-header/@subscribers").get()
    rank = selector.xpath("//strong[@id='position']/*/@number").get()    
    info["members"] = int(members) if members else None
    info["rank"] = int(rank) if rank else None
    info["bookmarks"] = {}
    for item in selector.xpath("//div[faceplate-tracker[@source='community_menu']]/faceplate-tracker"):
        name = item.xpath(".//a/span/span/span/text()").get()
        link = item.xpath(".//a/@href").get()
        info["bookmarks"][name] = link

    info["url"] = url
    post_data = []
    for box in selector.xpath("//article"):
        link = box.xpath(".//a/@href").get()
        author = box.xpath(".//shreddit-post/@author").get()
        post_label = box.xpath(".//faceplate-tracker[@source='post']/a/span/div/text()").get()
        upvotes = box.xpath(".//shreddit-post/@score").get()
        comment_count = box.xpath(".//shreddit-post/@comment-count").get()
        attachment_type = box.xpath(".//shreddit-post/@post-type").get()
        if attachment_type and attachment_type == "image":
            attachment_link = box.xpath(".//div[@slot='thumbnail']/*/*/@src").get()
        elif attachment_type == "video":
            attachment_link = box.xpath(".//shreddit-player/@preview").get()
        else:
            attachment_link = box.xpath(".//div[@slot='thumbnail']/a/@href").get()

        # Extract the paragraph from the specified div class
        paragraph = box.xpath(".//div[contains(@class, 'md feed-card-text-preview text-ellipsis line-clamp-3 xs:line-clamp-6 text-14')]//p/text()").get()

        post_data.append({
            "authorProfile": "https://www.reddit.com/user/" + author if author else None,
            "authorId": box.xpath(".//shreddit-post/@author-id").get(),
            "picture": box.xpath(".//shreddit-post/@content-href").get(),
            "title": box.xpath("./@aria-label").get(),
            "link": "https://www.reddit.com" + link if link else None,
            "publishingDate": box.xpath(".//shreddit-post/@created-timestamp").get(),
            "postId": box.xpath(".//shreddit-post/@id").get(),
            "postLabel": post_label.strip() if post_label else None,
            "postUpvotes": int(upvotes) if upvotes else None,
            "commentCount": int(comment_count) if comment_count else None,
            "attachmentType": attachment_type,
            "attachmentLink": attachment_link,
            "paragraph": paragraph.strip() if paragraph else None,
        })
    # id for the next posts batch
    cursor_id = selector.xpath("//shreddit-post/@more-posts-cursor").get()
    return {"post_data": post_data, "info": info, "cursor": cursor_id}

async def scrape_subreddit(subreddit_id: str, sort: Union["new", "hot", "old", "top"], max_pages: int = None):
    """scrape articles on a subreddit"""
    base_url = f"https://www.reddit.com/r/{subreddit_id}/"
    response = await client.get(base_url)
    subreddit_data = {}
    data = parse_subreddit(response)
    subreddit_data["info"] = data["info"]
    subreddit_data["posts"] = data["post_data"]
    cursor = data["cursor"]

    def make_pagination_url(cursor_id: str):
        return f"https://www.reddit.com/svc/shreddit/community-more-posts/top/?t=YEAR&after=dDNfMWVlMjNheA%3D%3D&name=cheating_stories&navigationSessionId=5dc61d06-c0f2-4474-a88c-27f426c7219e&feedLength=3&sort={sort}" 
        
    while cursor and (max_pages is None or max_pages > 0):
        url = make_pagination_url(cursor)
        response = await client.get(url)
        data = parse_subreddit(response)
        cursor = data["cursor"]
        post_data = data["post_data"]
        subreddit_data["posts"].extend(post_data)
        if max_pages is not None:
            max_pages -= 1
    log.success(f"scraped {len(subreddit_data['posts'])} posts from the rubreddit: r/{subreddit_id}")
    return subreddit_data

In [20]:
result = await scrape_subreddit( subreddit_id="cheating_stories",
        sort="top",
        max_pages=1)
print(result)

2025-01-13 20:53:09.203 | SUCCESS  | __main__:scrape_subreddit:78 - scraped 28 posts from the rubreddit: r/cheating_stories


{'info': {'id': 'cheating_stories?rdt=65485', 'description': 'Put your TRUE cheating stories here.', 'members': 254717, 'rank': None, 'bookmarks': {}, 'url': 'https://www.reddit.com/r/cheating_stories/?rdt=65485'}, 'posts': [{'authorProfile': 'https://www.reddit.com/user/Narutosmashedmymom', 'authorId': 't2_5fqdliu5', 'picture': 'https://www.reddit.com/r/cheating_stories/comments/1i0ih4d/pregnant_girlfriend_texted_her_ex/', 'title': 'Pregnant girlfriend texted her ex', 'link': 'https://www.reddit.com/r/cheating_stories/comments/1i0ih4d/pregnant_girlfriend_texted_her_ex/', 'publishingDate': '2025-01-13T16:54:33.094000+0000', 'postId': 't3_1i0ih4d', 'postLabel': None, 'postUpvotes': 26, 'commentCount': 69, 'attachmentType': 'text', 'attachmentLink': None, 'paragraph': 'I’ve seen this girl for awhile now, and I knocked her up pretty quick. Things are moving fast between us and it’s really been amazing. Me and her have discussed her ex before and she made it clear to me he was a shitty per

In [21]:
df = pd.DataFrame(result['posts'])

In [25]:
df.head(n=5)

,authorProfile,authorId,picture,title,link,publishingDate,postId,postLabel,postUpvotes,commentCount,attachmentType,attachmentLink,paragraph
0,https://www.reddit.com/user/Narutosmashedmymom,t2_5fqdliu5,https://www.reddit.com/r/cheating_stories/comm...,Pregnant girlfriend texted her ex,https://www.reddit.com/r/cheating_stories/comm...,2025-01-13T16:54:33.094000+0000,t3_1i0ih4d,None,26,69,text,None,"I’ve seen this girl for awhile now, and I knoc..."
1,https://www.reddit.com/user/Gold_Solution_5864,t2_aqjjvf6f,https://www.reddit.com/r/cheating_stories/comm...,My partner cheated on me and doesn’t know I’m ...,https://www.reddit.com/r/cheating_stories/comm...,2025-01-13T13:51:32.451000+0000,t3_1i0ee4n,None,25,22,text,None,I read my partner’s journal and found out they...
2,https://www.reddit.com/user/leytonscomet,t2_4lny63bh,https://www.reddit.com/r/cheating_stories/comm...,My husband’s new friendship is making me uncom...,https://www.reddit.com/r/cheating_stories/comm...,2025-01-12T20:48:14.477000+0000,t3_1hzwjon,None,200,198,text,None,I originally posted this on AITH but someone h...
3,https://www.reddit.com/user/InternationalPain965,t2_gi5xxwaw0,https://www.reddit.com/r/cheating_stories/comm...,My wife of 38 years has been cheating on me ne...,https://www.reddit.com/r/cheating_stories/comm...,2024-03-13T19:08:50.502000+0000,t3_1bdzv9f,None,558,300,text,None,Me 61 male and my wife 59 female have been mar...
4,https://www.reddit.com/user/Distribution_Initial,t2_78n8ql1b,https://www.reddit.com/r/cheating_stories/comm...,My husband's affair partner passed and it seem...,https://www.reddit.com/r/cheating_stories/comm...,2024-06-08T16:39:22.898000+0000,t3_1db70bt,None,495,301,text,None,"We've been married for over a decade, but at s..."
